Generates the dataset summary table 

In [16]:
import sys
sys.path.append('..')  
import pandas as pd
from src.data_loading import load_german, load_taiwan, load_folktables

In [ ]:


pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Semantic descriptions not available on the dataset object itself for
# StandardDataset-based loaders (Taiwan, Folktables) 

MANUAL_LABEL_MEANINGS = {
    'Taiwan Credit Default': {
        0.0: 'did not default (favourable)',
        1.0: 'defaulted (unfavourable)',
    },
    'Folktables ACS Employment': {
        1.0: 'employed (favourable)',
        0.0: 'not employed (unfavourable)',
    },
}
MANUAL_ATTR_MEANINGS = {
    'Taiwan Credit Default': {1.0: 'male (privileged)', 0.0: 'female (unprivileged)'},
    'Folktables ACS Employment': {1.0: 'male (privileged)', 0.0: 'female (unprivileged)'},
}


def describe_label(name, dataset, code):
    """Human-readable meaning of a favourable/unfavourable label code."""
    label_maps = dataset.metadata.get('label_maps')
    if label_maps:
        return label_maps[0].get(code, str(code))
    return MANUAL_LABEL_MEANINGS.get(name, {}).get(code, str(code))


def describe_protected_attrs(name, dataset):
    """
    Human-readable summary of each protected attribute: what 1 vs 0 means
    natively, e.g. 'sex (1=male/privileged, 0=female/unprivileged)'.
    """
    attr_maps = dataset.metadata.get('protected_attribute_maps')
    descriptions = []
    for i, attr in enumerate(dataset.protected_attribute_names):
        if attr_maps and i < len(attr_maps):
            mapping = attr_maps[i]
            parts = ', '.join(f'{k}={v}' for k, v in mapping.items())
        else:
            mapping = MANUAL_ATTR_MEANINGS.get(name, {})
            parts = ', '.join(f'{k}={v}' for k, v in mapping.items())
        descriptions.append(f'{attr} ({parts})' if parts else attr)
    return '; '.join(descriptions)


def summarize(name, dataset, source):
    n = dataset.features.shape[0]
    fav = dataset.favorable_label
    unfav = dataset.unfavorable_label
    return {
        'dataset': name,
        'n': n,
        'protected attribute(s)': describe_protected_attrs(name, dataset),
        'label': dataset.label_names[0],
        'favourable outcome': f'{fav} = {describe_label(name, dataset, fav)}',
        'unfavourable outcome': f'{unfav} = {describe_label(name, dataset, unfav)}',
        'source': source,
    }


rows = []

german = load_german()
rows.append(summarize(
        'German Credit', german,
        'AIF360 built-in (GermanDataset), UCI Statlog German Credit'
    ))

taiwan = load_taiwan()
rows.append(summarize(
        'Taiwan Credit Default', taiwan,
        'UCI ML Repository, ID 350'
    ))

folktables_data = load_folktables()
rows.append(summarize(
        'Folktables ACS Employment', folktables_data,
        'US Census ACS PUMS (2018, CA, 1-Year) via folktables package'
    ))

df = pd.DataFrame(rows).set_index('dataset')

print(df.to_string())
df.to_csv('../results/dataset_summary.csv')
print("\nSaved to results/dataset_summary.csv")


                                n                                  protected attribute(s)     label                  favourable outcome               unfavourable outcome                                                        source
dataset                                                                                                                                                                                                                                 
German Credit                1000    sex (1.0=Male, 0.0=Female); age (1.0=Old, 0.0=Young)    credit                   1.0 = Good Credit                   2.0 = Bad Credit    AIF360 built-in (GermanDataset), UCI Statlog German Credit
Taiwan Credit Default       30000  SEX (1.0=male (privileged), 0.0=female (unprivileged))   default  0.0 = did not default (favourable)     1.0 = defaulted (unfavourable)                                     UCI ML Repository, ID 350
Folktables ACS Employment  378817  SEX (1.0=male (privileged), 0.0=f

In [13]:
#print all columns for german dataset
german = load_german()
print(german.feature_names)

['month', 'credit_amount', 'investment_as_income_percentage', 'residence_since', 'age', 'number_of_credits', 'people_liable_for', 'sex', 'status=A11', 'status=A12', 'status=A13', 'status=A14', 'credit_history=A30', 'credit_history=A31', 'credit_history=A32', 'credit_history=A33', 'credit_history=A34', 'purpose=A40', 'purpose=A41', 'purpose=A410', 'purpose=A42', 'purpose=A43', 'purpose=A44', 'purpose=A45', 'purpose=A46', 'purpose=A48', 'purpose=A49', 'savings=A61', 'savings=A62', 'savings=A63', 'savings=A64', 'savings=A65', 'employment=A71', 'employment=A72', 'employment=A73', 'employment=A74', 'employment=A75', 'other_debtors=A101', 'other_debtors=A102', 'other_debtors=A103', 'property=A121', 'property=A122', 'property=A123', 'property=A124', 'installment_plans=A141', 'installment_plans=A142', 'installment_plans=A143', 'housing=A151', 'housing=A152', 'housing=A153', 'skill_level=A171', 'skill_level=A172', 'skill_level=A173', 'skill_level=A174', 'telephone=A191', 'telephone=A192', 'fore

In [14]:
print(taiwan.feature_names)

['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


In [15]:
print(folktables_data.feature_names)

['AGEP', 'SCHL', 'MAR', 'RELP', 'DIS', 'ESP', 'CIT', 'MIG', 'MIL', 'ANC', 'NATIVITY', 'DEAR', 'DEYE', 'DREM', 'SEX', 'RAC1P']
